# Animate toy-manifold tiling across epochs

Follow one MFA, ARD, or HDDC run from its **KMeans baseline**, through its saved
epochs, to its **saved best model**. The geometry plot shows components whose closest
manifold is the selected one. The assignments plot keeps the subset with
nonzero total assignments and valid PCs. Use **Play**, **Stop**, or
the checkpoint slider in each plot; Stop pauses that plot at its current frame.

There are two different PCA operations: the display PCA is fitted **once** on
the selected manifold; **each Gaussian's covariance PCs are computed again for
every checkpoint**. Frames are prepared before playback. Existing artifacts
are only read; missing assignments and KMeans covariances are computed in memory.

## 1. Choose a run and animation settings

Run this notebook from the repository root or `notebooks/`. Change `RUN_DIR` to inspect another run.

In [ ]:
from pathlib import Path

RUN_DIR = Path(
    "/u/dssc/zenocosini/decomposing-activations-local-geometry/dalg-cache/toy_manifolds_10types_1each_D128_30Keach_noise_sweep_seed0/noise_ratio_10/models/toy_noise10_sweepk_hddc_cluster_pca/hddc__toy_manifolds_10types_1each_d128_30keach_noise10__l00__k1000__q32__s42__9a50253d"
)
KMEANS_MODEL_PATH = ""  # None: use training.arguments.kmeans_model_path
MANIFOLD_ID = 2
N_COMPONENTS = 3          # 2: ellipses; 3: ellipsoids
EPOCH_START = None        # inclusive; None uses the first available epoch
EPOCH_END = None          # inclusive; None uses the last available epoch
EPOCH_STRIDE = 1          # subsample the sorted snapshots; retain the last one
INCLUDE_SAVED_BEST = True
FRAME_DURATION_MS = 500
FRAME_WORKERS = 4         # concurrent checkpoint/PCA workers; 1 runs sequentially

ASSIGNMENT_DEVICE = "cpu"  # change to "cuda" on a GPU node
ASSIGNMENT_BATCH_SIZE = 4096
MAX_PLOT_POINTS = 5000     # assignments and PCA still use all selected rows
ELLIPSE_N_STD = 0.8
KMEANS_COVARIANCE_FLOOR = 1e-8  # display only, for singular empirical covariances
POINT_SIZE = 4
POINT_ALPHA = 0.65
ELLIPSE_ALPHA = 0.10
ANNOTATE_COMPONENT_IDS = False
PLOT_WIDTH = 720
PLOT_PANEL_HEIGHT = 440
RANDOM_SEED = 0

## 2. Inspect the checkpoint timeline

In [ ]:
import json
import re
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import nullcontext
from threading import Lock

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.decomposition import PCA
from threadpoolctl import threadpool_limits
from torch.utils.data import DataLoader

REPO = Path.cwd().resolve()
if not (REPO / "src").is_dir():
    REPO = REPO.parent
if not (REPO / "src").is_dir():
    raise RuntimeError("Run from the repository root or notebooks/.")
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

from dalg.analysis.cluster_assignments import compute_assignments
from dalg.data.shard_activations import ActivationBatchDataset, load_meta_index
from dalg.data.subset_spec import resolve_spec_positions, split_shard_dir_spec
from dalg.evaluation.toy_manifold_tiling import _load_model
from dalg.models.kmeans import load_kmeans


def resolve_path(value):
    path = Path(value).expanduser()
    return (path if path.is_absolute() else REPO / path).resolve()


def has_model(directory):
    return (directory / "mfa_model.pt").is_file() or (directory / "mfa_model_shards.json").is_file()


def discover_checkpoints(run_dir, kmeans_path, start=None, end=None, stride=1, include_best=True):
    if stride < 1:
        raise ValueError("EPOCH_STRIDE must be positive.")
    entries = []
    for path in run_dir.glob("epoch_*"):
        match = re.fullmatch(r"epoch_(\d+)", path.name)
        if not match or not path.is_dir():
            continue
        epoch = int(match[1])
        if (start is None or epoch >= start) and (end is None or epoch <= end):
            if not has_model(path):
                raise FileNotFoundError(f"Incomplete epoch snapshot: {path}")
            entries.append((epoch, path))
    entries.sort(key=lambda entry: entry[0])
    if not entries:
        raise ValueError("No saved epoch snapshots in the selected range.")
    if len({epoch for epoch, _ in entries}) != len(entries):
        raise ValueError("Duplicate numeric epoch IDs.")
    selected = entries[::stride]
    if selected[-1] != entries[-1]:
        selected.append(entries[-1])

    timeline = [dict(name="kmeans", label="KMeans baseline", epoch=None,
                     kind="kmeans", model_path=kmeans_path)]
    timeline.extend(dict(name=f"epoch_{epoch}", label=f"Epoch {epoch}", epoch=epoch,
                         kind="epoch", model_path=path / "mfa_model.pt")
                    for epoch, path in selected)
    if include_best:
        if not has_model(run_dir):
            raise FileNotFoundError(f"No saved best model in {run_dir}")
        best_epoch = None
        for filename in ("checkpoint.pt", "checkpoint_rank0000.pt"):
            checkpoint_path = run_dir / filename
            if checkpoint_path.is_file():
                # Training checkpoints include optimizer and RNG state. Read only trusted runs.
                checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False, mmap=True)
                best_epoch = checkpoint.get("best_epoch")
                del checkpoint
                break
        label = "Saved best model" + (f" (epoch {best_epoch})" if best_epoch is not None else "")
        timeline.append(dict(name="saved_best", label=label, epoch=best_epoch,
                             kind="best", model_path=run_dir / "mfa_model.pt"))
    for entry in timeline:
        path = entry["model_path"]
        entry["assignments_path"] = path.with_name(f"{path.stem}_assignments.pt")
    return timeline


if not isinstance(FRAME_WORKERS, int) or FRAME_WORKERS < 1:
    raise ValueError("FRAME_WORKERS must be a positive integer.")
if N_COMPONENTS not in (2, 3):
    raise ValueError("N_COMPONENTS must be 2 or 3.")
for name in ("FRAME_DURATION_MS", "ASSIGNMENT_BATCH_SIZE", "MAX_PLOT_POINTS", "ELLIPSE_N_STD", "KMEANS_COVARIANCE_FLOOR"):
    if not np.isfinite(globals()[name]) or globals()[name] <= 0:
        raise ValueError(f"{name} must be finite and positive.")

RUN_DIR = resolve_path(RUN_DIR)
run_spec = json.loads((RUN_DIR / "run_spec.json").read_text())
model_kind = run_spec["training"]["model_kind"]
if model_kind not in {"mfa", "ard", "hddc"}:
    raise ValueError(f"Expected an MFA, ARD, or HDDC run, got {model_kind!r}.")
initialization_path = KMEANS_MODEL_PATH or run_spec["training"]["arguments"].get("kmeans_model_path")
if not initialization_path:
    raise ValueError("Set KMEANS_MODEL_PATH to this run's original KMeans model checkpoint.")
kmeans_path = resolve_path(initialization_path)
if not kmeans_path.is_file():
    raise FileNotFoundError(f"Missing KMeans initialization checkpoint: {kmeans_path}")
timeline = discover_checkpoints(RUN_DIR, kmeans_path, EPOCH_START, EPOCH_END,
                                EPOCH_STRIDE, INCLUDE_SAVED_BEST)
display(pd.DataFrame([dict(
    frame=i, label=entry["label"], epoch=entry["epoch"],
    model_path=str(entry["model_path"]),
    saved_assignments=entry["assignments_path"].is_file(),
) for i, entry in enumerate(timeline)]))
print("Saved best is shown separately because training can restore an earlier epoch.")

## 3. Load the canonical point stream and select a manifold

The table lists planted manifold instances. If needed, change `MANIFOLD_ID`
above and rerun from this section. Stream metadata is checked before aligning
manifold labels or assignment vectors.

In [ ]:
dataset_spec = run_spec["dataset"]
shard_dir, subset_spec = split_shard_dir_spec(dataset_spec["shard_dir"])
shard_dir = resolve_path(shard_dir)
layer = int(dataset_spec["layer"])
shard_config = json.loads((shard_dir / "config.json").read_text())
if shard_config.get("source_kind") != "toy_manifolds":
    raise ValueError("The run must reference toy-manifold activation shards.")
window, drop_prefix = int(shard_config["window"]), int(shard_config.get("drop_prefix", 0))
if window != 1 or drop_prefix != 0:
    raise ValueError("Expected one toy activation per row (window=1, drop_prefix=0).")
manifold_metadata = torch.load(shard_dir / shard_config["manifold_metadata"],
                               map_location="cpu", weights_only=True)
all_manifold_ids = manifold_metadata["row_manifold_ids"].reshape(-1).long()
meta_index = load_meta_index(shard_dir, layer=layer)
if len(meta_index) != len(all_manifold_ids):
    raise ValueError("Manifold labels and activation metadata have different lengths.")
positions = resolve_spec_positions(meta_index, subset_spec, window=window, drop_prefix=drop_prefix)
expected_rows = torch.tensor([meta_index[i]["global_row"] for i in positions])
if len(expected_rows) == 0 or expected_rows.min() < 0 or expected_rows.max() >= len(all_manifold_ids):
    raise ValueError("No selected rows or invalid global row IDs.")
row_manifold_ids = all_manifold_ids[expected_rows]
dataset = ActivationBatchDataset(
    shard_dir, layer=layer, row_subset=positions, batch_size=ASSIGNMENT_BATCH_SIZE,
    drop_prefix=drop_prefix, dtype=torch.float32, shuffle_shards=False,
    shuffle_within_shard=False, seed=0, return_metadata=True,
)
chunks = []
offset = 0
for points, rows, token_positions in DataLoader(dataset, batch_size=None, num_workers=0):
    if not torch.equal(rows, expected_rows[offset:offset + len(rows)]) or token_positions.any():
        raise ValueError("Activation batches are not in the expected canonical row order.")
    chunks.append(points)
    offset += len(rows)
all_points = torch.cat(chunks).cpu()
del chunks
D = int(shard_config["d_model"])
if all_points.shape != (len(positions), D) or not torch.isfinite(all_points).all():
    raise ValueError("Invalid activation shape or non-finite points.")
manifold_counts = torch.bincount(row_manifold_ids, minlength=int(manifold_metadata["num_manifolds"]))
manifold_table = pd.DataFrame([dict(
    manifold_id=int(item["manifold_id"]), type_name=item["type_name"],
    intrinsic_dim=int(item["intrinsic_dim"]), point_count=int(manifold_counts[int(item["manifold_id"])]),
) for item in manifold_metadata["manifolds"]])
display(manifold_table)
selected = manifold_table[manifold_table.manifold_id == MANIFOLD_ID]
if selected.empty or int(selected.iloc[0].point_count) == 0:
    raise ValueError(f"No points for manifold {MANIFOLD_ID}.")
manifold_name = selected.iloc[0].type_name
manifold_mask = row_manifold_ids == MANIFOLD_ID
manifold_points = all_points[manifold_mask].contiguous()

# The display coordinate system is fitted once, independently of model checkpoints.
pca = PCA(n_components=N_COMPONENTS, svd_solver="full")
projected_points = pca.fit_transform(manifold_points.numpy()).astype(np.float64)
projection = pca.components_.astype(np.float64)
rng = np.random.default_rng(RANDOM_SEED)
shown = np.sort(rng.choice(len(manifold_points), min(MAX_PLOT_POINTS, len(manifold_points)), replace=False))
shown_points = projected_points[shown]
display(pd.DataFrame({"display_PC": np.arange(1, N_COMPONENTS + 1),
                      "explained_variance_ratio": pca.explained_variance_ratio_}))
print(f"Loaded {len(all_points):,} points; displaying {len(shown):,} from {manifold_name}.")

## 4. Prepare assignments and Gaussian PCs for each checkpoint

For every Gaussian checkpoint, construct $\Sigma_k=W_kW_k^\top+\Psi_k$ and
eigendecompose it in the original $D$ dimensions. The HDDC model's `_W()`
includes its saved rank mask; the noise remains present. Project the leading
ambient eigenvectors into the fixed display PCA coordinates. The displayed
covariances are exactly $P\Sigma_kP^\top$.

The **KMeans baseline** uses population covariances around assigned empirical
means, with footprints centered at the saved centroids, as in the KMeans tiling
notebook. These describe cluster spread; they are not an epoch-zero Gaussian
model. No KMeans rank truncation is applied. Covariance footprints are omitted
for clusters with fewer than two points; singular empirical covariances receive
the configured display floor after projection.

Assignments cover all selected dataset rows, including other manifolds, so
hover counts and purity use the full partition. Existing assignment bundles
must match this checkpoint and stream; missing bundles are computed in memory.

`FRAME_WORKERS` controls concurrent checkpoint preparation (default: 4; set 1
for sequential execution). The KMeans baseline is prepared first to establish
K, then workers prepare the epochs and saved best model. Results retain timeline
order even when workers finish out of order. Workers share the loaded points;
each holds its own model and ambient covariances, so more workers use more RAM.
During preparation, PyTorch and BLAS use one CPU thread per worker. Missing
assignments on an accelerator are computed one checkpoint at a time while CPU
PCA work can continue in other workers.

### Which components appear?

- **Geometry:** the selected manifold must be the component mean's closest
  planted manifold. The existing evaluator computes the distance to each
  noiseless manifold using its saved embedding and calibration. There is no
  distance cutoff and no assignment requirement.
- **Assignments:** the same closest-manifold association, plus at least one
  assigned point in the full selected dataset and valid covariance PCs. For
  MFA/ARD/HDDC these are the saved Gaussian's covariance PCs. For the KMeans
  baseline, empirical PCA requires at least two assigned points. Components
  without KMeans PCA remain centroid markers in the geometry plot only.

Both selections are recomputed at every checkpoint. A distance tie between
different manifolds is reported as ambiguous and excluded, following the
evaluator's association contract. Point colors still show the hard assignments
of every displayed manifold point, including assignments crossing manifold
associations; hover reports the component's closest manifold.


In [ ]:
from dalg.evaluation.toy_manifold_metrics import _associate_component_means

# Share CPU data across workers; keep only one assignment model on the accelerator.
assignment_device_lock = Lock()


def validate_assignments(labels, counts, K, num_items):
    if labels.dtype not in (torch.int32, torch.int64) or counts.dtype not in (torch.int32, torch.int64):
        raise ValueError("Assignment IDs and counts must be integers.")
    if labels.shape != (num_items,) or counts.shape != (K,):
        raise ValueError("Assignment vector or cluster count shape mismatch.")
    if labels.numel() and (labels.min() < 0 or labels.max() >= K):
        raise ValueError("Assignment IDs outside [0, K).")
    if not torch.equal(torch.bincount(labels.long(), minlength=K), counts.long()):
        raise ValueError("Cluster counts do not match assignments.")


def checkpoint_assignments(entry, K):
    assignment_path = entry["assignments_path"]
    assignment_type = "kmeans" if entry["kind"] == "kmeans" else ("hddc" if model_kind == "hddc" else "mfa")
    if assignment_path.is_file():
        bundle = torch.load(assignment_path, map_location="cpu", weights_only=True, mmap=True)
        if int(bundle["K"]) != K or bundle.get("subset_spec") != subset_spec:
            raise ValueError(f"Assignment K/subset mismatch: {assignment_path}")
        if bundle.get("model_type") not in (None, assignment_type):
            raise ValueError(f"Assignment model type mismatch: {assignment_path}")
        if not bundle.get("model_path") or resolve_path(bundle["model_path"]) != entry["model_path"].resolve():
            raise ValueError(f"Assignment checkpoint provenance mismatch: {assignment_path}")
        source = bundle.get("source", {})
        expected = dict(layer=layer, drop_prefix=drop_prefix, num_items=len(all_points))
        if not source.get("shard_dir") or resolve_path(source["shard_dir"]) != shard_dir:
            raise ValueError(f"Assignment shard provenance mismatch: {assignment_path}")
        if any(source.get(key) != value for key, value in expected.items()):
            raise ValueError(f"Assignment stream provenance mismatch: {assignment_path}")
        labels, counts = bundle["assignments"], bundle["cluster_sizes"]
        origin = "saved"
    else:
        assignment_guard = (assignment_device_lock if torch.device(ASSIGNMENT_DEVICE).type != "cpu"
                            else nullcontext())
        with assignment_guard:
            counts, labels, _, _ = compute_assignments(
                entry["model_path"], all_points.split(ASSIGNMENT_BATCH_SIZE),
                device=ASSIGNMENT_DEVICE, model_type=assignment_type,
            )
        origin = "computed in memory"
    validate_assignments(labels, counts, K, len(all_points))
    return labels.long(), counts.long(), origin


def empirical_covariances(points, labels, means, counts):
    # Group once, then reuse the empirical covariance calculation from the KMeans notebook.
    order = torch.argsort(labels)
    offsets = torch.cat([torch.zeros(1, dtype=torch.long), counts.cumsum(0)])
    covariances = torch.zeros(len(means), points.shape[1], points.shape[1], dtype=torch.float64)
    valid = counts >= 2
    for component in valid.nonzero().flatten().tolist():
        cluster = points[order[offsets[component]:offsets[component + 1]]].double()
        residual = cluster - cluster.mean(dim=0)
        covariances[component] = residual.T @ residual / len(cluster)
    return covariances, valid


def project_gaussians(means, covariances, basis, center, covariance_floor=0.0):
    # These are the Gaussian's ambient PCs, recomputed at EVERY checkpoint.
    values, vectors = torch.linalg.eigh(covariances.double())
    n = basis.shape[0]
    ambient_values = values[:, -n:].flip(-1)
    ambient_vectors = vectors[:, :, -n:].flip(-1)
    if not torch.isfinite(values).all() or values.min() < -1e-8:
        raise ValueError("Invalid ambient covariance eigenspectrum.")
    cov = np.einsum("pd,kde,qe->kpq", basis, covariances.numpy(), basis, optimize=True)
    cov = 0.5 * (cov + cov.transpose(0, 2, 1))
    if covariance_floor:
        vals, vecs = np.linalg.eigh(cov)
        cov = (vecs * np.maximum(vals, covariance_floor)[:, None, :]) @ vecs.transpose(0, 2, 1)
    elif np.linalg.eigvalsh(cov).min() <= 0:
        raise ValueError("Projected Gaussian covariance is not positive definite.")
    projected_means = (means.numpy() - center) @ basis.T
    directions = np.einsum("pd,kdr->krp", basis, ambient_vectors.numpy(), optimize=True)
    if not all(np.isfinite(array).all() for array in (projected_means, cov, directions)):
        raise ValueError("Non-finite projected geometry.")
    return projected_means, cov, directions, ambient_values.numpy()


def component_plot_masks(means, metadata, manifold_id, counts, pca_valid):
    # Use distances to the planted manifolds, independently of sampled points and assignments.
    association = _associate_component_means(means, metadata, max_mean_to_manifold_distance=None)
    manifold_ids = torch.tensor([int(item["manifold_id"]) for item in metadata["manifolds"]])
    nearest_ids = torch.full((len(means),), -1, dtype=torch.long)
    associated = association.manifold_indices >= 0
    nearest_ids[associated] = manifold_ids[association.manifold_indices[associated]]
    geometry_mask = nearest_ids == manifold_id
    assignments_mask = geometry_mask & (counts > 0) & pca_valid
    return (geometry_mask, assignments_mask), nearest_ids, association


def prepare_frame(entry, expected_K=None):
    current = (load_kmeans(entry["model_path"], map_location="cpu") if entry["kind"] == "kmeans"
               else _load_model(entry["model_path"].parent, model_kind)).cpu().eval()
    K = current.K
    if current.D != D or (expected_K is not None and K != expected_K):
        raise ValueError(f"Checkpoint K/D mismatch: {entry['model_path']}")
    labels, counts, origin = checkpoint_assignments(entry, K)
    means = current.mu.detach().cpu().double()
    with torch.no_grad():
        if entry["kind"] == "kmeans":
            covariances, valid = empirical_covariances(all_points, labels, means, counts)
            weights = counts.double() / len(labels)
            ranks = np.full(K, np.nan)
        else:
            W, psi = current._W().double(), current._psi().double()
            covariances = W @ W.transpose(1, 2) + torch.diag_embed(psi)
            valid = torch.ones(K, dtype=torch.bool)
            weights = current.pi_logits.softmax(0).double()
            if model_kind == "hddc":
                ranks = current.rank_mask.sum(1).cpu().numpy()
            else:
                threshold = float(getattr(current, "rank_threshold", 1.0))
                ranks = (current._scale().square() > threshold * psi.mean(1, keepdim=True)).sum(1).cpu().numpy()
        projected_means, projected_cov, directions, pc_values = project_gaussians(
            means, covariances, projection, pca.mean_,
            KMEANS_COVARIANCE_FLOOR if entry["kind"] == "kmeans" else 0.0,
        )
    selected_labels = labels[manifold_mask]
    points_here = torch.bincount(selected_labels, minlength=K)
    masks, nearest_ids, association = component_plot_masks(
        means, manifold_metadata, MANIFOLD_ID, counts, valid,
    )
    # Keep only projected geometry and the displayed point labels for playback.
    return dict(
        name=entry["name"], label=entry["label"], K=K, kind=entry["kind"],
        means=projected_means, covariances=projected_cov, pc_directions=directions,
        pc_values=pc_values, valid=valid.numpy(), masks=tuple(mask.numpy() for mask in masks),
        shown_labels=selected_labels[shown].numpy(), counts=counts.numpy(),
        points_here=points_here.numpy(), distances=association.nearest_distances.numpy(), ranks=ranks,
        nearest_manifold_ids=nearest_ids.numpy(), ambiguous=association.ambiguous.numpy(),
        weights=weights.detach().cpu().numpy(), assignment_origin=origin,
    )


def prepare_frames(entries, workers):
    states = [None] * len(entries)
    workers = min(workers, max(1, len(entries) - 1))
    previous_threads = torch.get_num_threads()
    try:
        # Bound native math threads so checkpoint workers do not oversubscribe the CPU.
        torch.set_num_threads(1)
        with threadpool_limits(limits=1):
            print(f"Preparing 1/{len(entries)}: {entries[0]['label']}")
            states[0] = prepare_frame(entries[0])
            print(f"Preparing {len(entries) - 1} checkpoints with {workers} worker(s)")
            with ThreadPoolExecutor(max_workers=workers) as pool:
                futures = {
                    pool.submit(prepare_frame, entry, states[0]["K"]): index
                    for index, entry in enumerate(entries[1:], start=1)
                }
                for completed, future in enumerate(as_completed(futures), start=2):
                    index = futures[future]
                    states[index] = future.result()
                    print(f"Prepared {completed}/{len(entries)}: {entries[index]['label']}")
    finally:
        torch.set_num_threads(previous_threads)
    return states


frame_states = prepare_frames(timeline, FRAME_WORKERS)
K = frame_states[0]["K"]
display(pd.DataFrame([dict(
    checkpoint=state["label"], closest_manifold_components=int(state["masks"][0].sum()),
    assigned_pca_valid_components=int(state["masks"][1].sum()),
    ambiguous_components=int(state["ambiguous"].sum()), assignments=state["assignment_origin"],
) for state in frame_states]))

## 5. Shared animation helpers

Solid, dashed, and dotted segments show ambient covariance PC1, PC2, and PC3
projected into the fixed display coordinates. Segments are clipped to their
footprints. KMeans hover weights are cluster fractions; Gaussian weights are
mixture probabilities. HDDC ranks use the saved mask; MFA/ARD ranks use the
loading-to-noise threshold. KMeans has no Gaussian rank.

Axis limits include all displayed footprints across the timeline. Point
coordinates and component colors stay fixed; point colors follow each frame's
assignments. Geometry uses closest-manifold association; the assignments plot
restricts that population to assigned components with valid PCs. Each plot
has independent playback and camera controls.

In [ ]:
import plotly.graph_objects as go
from plotly.colors import sample_colorscale

COMPONENT_COLORS = sample_colorscale("Turbo", np.linspace(0.02, 0.98, K))


def component_color(component_id, alpha=None):
    color = COMPONENT_COLORS[int(component_id)]
    return color if alpha is None else color.replace("rgb(", "rgba(").replace(")", f", {alpha})")


def covariance_eigendecomposition(covariance):
    covariance = np.asarray(covariance, dtype=np.float64)
    covariance = 0.5 * (covariance + covariance.T)
    values, vectors = np.linalg.eigh(covariance)
    if values.min() <= 0:
        raise ValueError("Projected covariance is not positive definite.")
    return covariance, values, vectors


def covariance_precision(covariance):
    covariance, values, vectors = covariance_eigendecomposition(covariance)
    precision = (vectors / values[None, :]) @ vectors.T
    return covariance, precision


def ellipse_polygon(mean, covariance, n_std, num_points=100):
    mean = np.asarray(mean, dtype=np.float64)
    _, values, vectors = covariance_eigendecomposition(covariance)
    transform = vectors @ np.diag(n_std * np.sqrt(values))
    angles = np.linspace(0.0, 2.0 * np.pi, num_points)
    circle = np.stack([np.cos(angles), np.sin(angles)])
    return mean[:, None] + transform @ circle


def ellipsoid_mesh(mean, covariance, n_std, num_u=18, num_v=10):
    mean = np.asarray(mean, dtype=np.float64)
    _, values, vectors = covariance_eigendecomposition(covariance)
    transform = vectors @ np.diag(n_std * np.sqrt(values))
    u = np.linspace(0.0, 2.0 * np.pi, num_u, endpoint=False)
    v = np.linspace(0.0, np.pi, num_v)
    uu, vv = np.meshgrid(u, v)
    sphere = np.stack([
        np.cos(uu) * np.sin(vv),
        np.sin(uu) * np.sin(vv),
        np.cos(vv),
    ]).reshape(3, -1)
    vertices = mean[:, None] + transform @ sphere
    face_i, face_j, face_k = [], [], []
    for row_index in range(num_v - 1):
        for column_index in range(num_u):
            next_column = (column_index + 1) % num_u
            top_left = row_index * num_u + column_index
            top_right = row_index * num_u + next_column
            bottom_left = (row_index + 1) * num_u + column_index
            bottom_right = (row_index + 1) * num_u + next_column
            face_i.extend([top_left, top_right])
            face_j.extend([top_right, bottom_right])
            face_k.extend([bottom_left, bottom_left])
    return vertices, face_i, face_j, face_k


def clipped_axis_segment(mean, covariance, direction, n_std):
    mean = np.asarray(mean, dtype=np.float64)
    direction = np.asarray(direction, dtype=np.float64)
    direction_norm = np.linalg.norm(direction)
    if direction_norm < 1e-12:
        return np.stack([mean, mean])
    unit = direction / direction_norm
    _, precision = covariance_precision(covariance)
    half_length = n_std / np.sqrt(max(float(unit @ precision @ unit), 1e-12))
    return np.stack([mean - half_length * unit, mean + half_length * unit])

In [ ]:
TRACE_CLASS = go.Scatter if N_COMPONENTS == 2 else go.Scatter3d
PC_STYLES = [("solid", "#222222"), ("dash", "#d23737"), ("dot", "#235fd2")]


def coordinates(points):
    args = dict(x=points[:, 0].tolist(), y=points[:, 1].tolist())
    if N_COMPONENTS == 3:
        args["z"] = points[:, 2].tolist()
    return args


def component_details(state, ids):
    ids = np.asarray(ids, dtype=int)
    return np.column_stack([
        ids, state["points_here"][ids], state["counts"][ids], state["distances"][ids],
        state["weights"][ids], state["ranks"][ids],
        state["points_here"][ids] / np.maximum(state["counts"][ids], 1),
        state["nearest_manifold_ids"][ids],
    ])


COMPONENT_HOVER = (
    "component %{customdata[0]:.0f}<br>points on manifold %{customdata[1]:,.0f}<br>"
    "total assigned points %{customdata[2]:,.0f}<br>closest manifold %{customdata[7]:.0f}<br>distance to closest manifold %{customdata[3]:.5f}<br>"
    "weight / fraction %{customdata[4]:.5f}<br>Gaussian effective rank %{customdata[5]:.0f}<br>"
    "manifold purity %{customdata[6]:.3f}<extra></extra>"
)


def footprint_trace(state, panel, component):
    present = state["masks"][panel][component] and state["valid"][component]
    mean, cov = state["means"][component], state["covariances"][component]
    if N_COMPONENTS == 2:
        vertices = ellipse_polygon(mean, cov, ELLIPSE_N_STD).T if present else np.empty((0, 2))
        trace = go.Scatter(
            **coordinates(vertices), mode="lines", fill="toself",
            line=dict(color=component_color(component, 0.65), width=1),
            fillcolor=component_color(component, ELLIPSE_ALPHA),
        )
    else:
        if present:
            vertices, i, j, k = ellipsoid_mesh(mean, cov, ELLIPSE_N_STD)
            vertices = vertices.T
        else:
            vertices, i, j, k = np.empty((0, 3)), [], [], []
        trace = go.Mesh3d(**coordinates(vertices), i=i, j=j, k=k,
                          color=component_color(component), opacity=ELLIPSE_ALPHA, showscale=False)
    trace.update(customdata=np.repeat(component_details(state, [component]), len(vertices), axis=0).tolist(),
                 hovertemplate=COMPONENT_HOVER, showlegend=False, name=f"Component {component}")
    return trace


def axis_trace(state, panel, pc):
    ids = np.flatnonzero(state["masks"][panel] & state["valid"])
    segments = []
    for component in ids:
        segment = clipped_axis_segment(state["means"][component], state["covariances"][component],
                                       state["pc_directions"][component, pc], ELLIPSE_N_STD)
        segments.extend([*segment, np.full(N_COMPONENTS, np.nan)])
    points = np.asarray(segments).reshape(-1, N_COMPONENTS)
    dash, color = PC_STYLES[pc]
    return TRACE_CLASS(**coordinates(points), mode="lines", connectgaps=False,
                       line=dict(dash=dash, color=color, width=2), hoverinfo="skip",
                       name=f"ambient PC{pc + 1}", showlegend=True, legendgroup=f"pc{pc}")


def mean_trace(state, panel):
    ids = np.flatnonzero(state["masks"][panel])
    return TRACE_CLASS(
        **coordinates(state["means"][ids]),
        mode="markers+text" if ANNOTATE_COMPONENT_IDS else "markers",
        text=[str(k) for k in ids] if ANNOTATE_COMPONENT_IDS else [], textposition="top center",
        marker=dict(size=7, symbol="x", color=[component_color(k) for k in ids]),
        customdata=component_details(state, ids).tolist(), hovertemplate=COMPONENT_HOVER,
        showlegend=False, name="Centroids",
    )


def point_update(state):
    ids = state["shown_labels"]
    # Frames intentionally omit coordinates and visibility, preserving the fixed cloud and toggle.
    return TRACE_CLASS(marker=dict(color=[component_color(k) for k in ids]),
                       customdata=component_details(state, ids).tolist())


def frame_title(state, panel):
    view_name = "Geometry" if panel == 0 else "Assignments"
    interpretation = "empirical cluster spread" if state["kind"] == "kmeans" else "full Gaussian covariance"
    count_label = "closest" if panel == 0 else "eligible"
    return dict(text=(f"{view_name} · {model_kind.upper()} · manifold {MANIFOLD_ID} ({manifold_name})"
                      f"<br>{state['label']}<br><sup>{interpretation}; {ELLIPSE_N_STD:g}σ footprints; "
                      f"{int(state['masks'][panel].sum())} {count_label} components</sup>"),
                x=0.5, xanchor="center", y=0.985, yanchor="top", yref="container")


def build_animation(states, panel):
    if panel not in (0, 1):
        raise ValueError("panel must be 0 (geometry) or 1 (assignments).")
    fig = go.Figure()
    fig.add_trace(TRACE_CLASS(
        **coordinates(shown_points), mode="markers", name="Dataset points", showlegend=False,
        marker=dict(size=POINT_SIZE, opacity=POINT_ALPHA, color="#555555"),
        hovertemplate=COMPONENT_HOVER if panel else "(%{x:.4f}, %{y:.4f})<extra></extra>",
    ))
    # A fixed slot for every component that appears at least once in this view.
    components = np.flatnonzero(np.logical_or.reduce([s["masks"][panel] for s in states]))
    slots = [("footprint", int(k)) for k in components]
    slots.extend(("axis", pc) for pc in range(N_COMPONENTS))
    slots.append(("mean", None))

    def updates(state):
        traces = [point_update(state)] if panel == 1 else []
        for kind, item in slots:
            trace = (footprint_trace(state, panel, item) if kind == "footprint" else
                     axis_trace(state, panel, item) if kind == "axis" else mean_trace(state, panel))
            traces.append(trace)
        return traces

    first_updates = updates(states[0])
    if panel == 1:
        fig.data[0].update(first_updates[0].to_plotly_json())
    fig.add_traces(first_updates[1:] if panel == 1 else first_updates)
    dynamic_indices = list(range(0 if panel == 1 else 1, len(fig.data)))
    fig.frames = [go.Frame(name=s["name"], data=updates(s), traces=dynamic_indices,
                           layout=dict(title=frame_title(s, panel))) for s in states]

    # Both plots use the same fixed limits for comparison across views and epochs.
    low, high = projected_points.min(0), projected_points.max(0)
    for state in states:
        visible = state["masks"][0] | state["masks"][1]
        means = state["means"][visible]
        radii = ELLIPSE_N_STD * np.sqrt(np.diagonal(state["covariances"][visible], axis1=1, axis2=2))
        radii[~state["valid"][visible]] = 0
        if len(means):
            low = np.minimum(low, (means - radii).min(0))
            high = np.maximum(high, (means + radii).max(0))
    margin = np.maximum(0.08 * (high - low), 1e-6)
    ranges = np.stack([low - margin, high + margin], axis=1)
    if N_COMPONENTS == 2:
        fig.update_xaxes(range=ranges[0], autorange=False, title_text="display PCA 1")
        fig.update_yaxes(range=ranges[1], autorange=False, title_text="display PCA 2",
                         scaleanchor="x", scaleratio=1)
    else:
        fig.update_layout(scene=dict(
            **{f"{axis}axis": dict(range=ranges[j], autorange=False, title=f"display PCA {j + 1}")
               for j, axis in enumerate("xyz")}, aspectmode="data", uirevision="fixed-camera",
        ))
    jump = dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))
    fig.update_layout(
        template="plotly_white", width=PLOT_WIDTH, height=PLOT_PANEL_HEIGHT + 280,
        title=frame_title(states[0], panel), uirevision="fixed-view", hovermode="closest",
        margin=dict(l=55, r=35, t=190, b=90),
        legend=dict(orientation="h", x=0.5, xanchor="center", y=1.045, yanchor="bottom"),
        updatemenus=[
            dict(type="buttons", direction="right", showactive=False, x=0, y=-0.08,
                 buttons=[
                     dict(label="Play", method="animate", args=[None, dict(
                         mode="immediate", fromcurrent=True, transition=dict(duration=0),
                         frame=dict(duration=FRAME_DURATION_MS, redraw=True))]),
                     dict(label="Stop", method="animate", args=[[None], jump]),
                 ]),
            dict(type="buttons", direction="right", showactive=False, x=0, y=1.17, yanchor="bottom",
                 buttons=[
                     dict(label="Show dataset points", method="restyle", args=[dict(visible=True), [0]]),
                     dict(label="Gaussians only", method="restyle", args=[dict(visible=False), [0]]),
                 ]),
        ],
        sliders=[dict(active=0, x=0.23, len=0.77, y=-0.08, currentvalue=dict(prefix="Checkpoint: "),
                      steps=[dict(label=s["label"], method="animate", args=[[s["name"]], jump]) for s in states])],
    )
    return fig


## 6. Geometry: components closest to this manifold

In [ ]:
geometry_animation = build_animation(frame_states, panel=0)
geometry_animation.show(config=dict(scrollZoom=True, responsive=True, displaylogo=False))

## 7. Assignments: closest components with assignments and valid PCs

In [ ]:
assignments_animation = build_animation(frame_states, panel=1)
assignments_animation.show(config=dict(scrollZoom=True, responsive=True, displaylogo=False))

## Reading and using the animation

- **Play** advances from the current checkpoint. **Stop** pauses there; use the
  slider to inspect any frame or return to the KMeans baseline after playback.
- Drag to pan/orbit and scroll to zoom. The two plots have independent playback and cameras;
  playback preserves each view. **Gaussians only** hides that plot's point cloud until
  **Show dataset points** is pressed, including when seeking to other frames.
- The KMeans baseline summarizes empirical spread around assigned empirical
  means, displayed at saved centroids. Its jump to epoch 1 also includes the
  change to a learned Gaussian covariance with noise.
- A saved best model can return to an earlier training epoch. It is a separately
  labeled final frame; the last numbered frame is the last selected saved epoch.
- Assignments are recomputed only when their checkpoint has no saved bundle.
  Increasing `EPOCH_STRIDE` reduces preparation and browser memory; reducing
  `MAX_PLOT_POINTS` reduces rendering cost without changing assignments or PCs.
- Near-degenerate covariance eigenvalues can make individual PC directions
  unstable even when the covariance footprint barely changes.

All controls work within the rendered Plotly figure, without widget extensions.
Rerun the frame-preparation and plotting cells after changing geometry settings.
Rerun from the data-loading section after changing manifold or display dimension.